In [ ]:
!pip install pyspark

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Mall_Customers.csv to Mall_Customers.csv


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ECommerceAnalytics") \
    .master("local[*]") \
    .getOrCreate()

print("Spark version:", spark.version)

In [ ]:
df = spark.read.csv(
    "data.csv",
    header=True,
    inferSchema=True
)

df.show(5)

In [ ]:
print("Total records:", df.count())

In [ ]:
df.printSchema()

In [ ]:
from pyspark.sql.functions import col

clean_df = df.filter(
    (col("Quantity") > 0) &
    (col("UnitPrice") > 0) &
    col("CustomerID").isNotNull() &
    col("Description").isNotNull()
)

print("Original records:", df.count())
print("Cleaned records:", clean_df.count())

In [ ]:
clean_df = clean_df.withColumn(
    "Revenue",
    col("Quantity") * col("UnitPrice")
)

clean_df.select(
    "InvoiceNo",
    "CustomerID",
    "Description",
    "Quantity",
    "UnitPrice",
    "Revenue"
).show(10)

In [ ]:
from pyspark.sql.functions import sum as spark_sum

customer_product = clean_df.groupBy(
    "CustomerID", "StockCode"
).agg(
    spark_sum("Quantity").alias("Quantity")
)

customer_product.show(10)

In [ ]:
import pandas as pd

cp_pd = customer_product.toPandas()

print("Rows:", len(cp_pd))
cp_pd.head()

In [ ]:
customer_product_matrix = cp_pd.pivot_table(
    index="CustomerID",
    columns="StockCode",
    values="Quantity",
    fill_value=0
)

print("Matrix shape:", customer_product_matrix.shape)

In [ ]:
!pip install scikit-learn

In [ ]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(
    n_neighbors=6,
    metric="cosine",
    algorithm="brute"
)

knn.fit(customer_product_matrix.values)

In [ ]:
customer_id = customer_product_matrix.index[0]

customer_index = customer_product_matrix.index.get_loc(customer_id)

distances, indices = knn.kneighbors(
    customer_product_matrix.iloc[customer_index].values.reshape(1, -1)
)

similar_customers = customer_product_matrix.iloc[
    indices[0][1:]
].index

print("Target Customer:", customer_id)
print("Similar Customers:", list(similar_customers))

In [ ]:
target_products = set(
    customer_product_matrix.loc[customer_id]
    [customer_product_matrix.loc[customer_id] > 0].index
)

recommended_scores = {}

for customer in similar_customers:
    products = customer_product_matrix.loc[customer]

    for product, quantity in products[products > 0].items():
        if product not in target_products:
            recommended_scores[product] = (
                recommended_scores.get(product, 0) + quantity
            )

recommendations = sorted(
    recommended_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

print("Recommended Products:")
for product, score in recommendations:
    print(product, "→ Score:", score)

In [ ]:
product_names = (
    clean_df
    .select("StockCode", "Description")
    .dropna()
    .dropDuplicates(["StockCode"])
    .toPandas()
)

product_names.head()

In [ ]:
product_lookup = dict(
    zip(product_names["StockCode"], product_names["Description"])
)

In [ ]:
print("Target Customer:", customer_id)
print("\nRecommended Products:\n")

for product, score in recommendations:
    description = product_lookup.get(product, "Unknown Product")
    print(f"{description} ({product}) → Score: {score}")

In [ ]:
recommendation_df = pd.DataFrame(
    recommendations,
    columns=["StockCode", "Score"]
)

recommendation_df["Product"] = recommendation_df["StockCode"].map(
    product_lookup
)

recommendation_df = recommendation_df[
    ["StockCode", "Product", "Score"]
]

recommendation_df

In [ ]:
import matplotlib.pyplot as plt

plot_df = recommendation_df.head(10).copy()

plt.figure(figsize=(12, 6))

plt.barh(
    plot_df["Product"].astype(str),
    plot_df["Score"]
)

plt.xlabel("Recommendation Score")
plt.ylabel("Product")
plt.title(f"Top Recommended Products for Customer {customer_id}")

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()